**MGMT298D: Science and Strategy of AI**
# Week 5: Transfer Learning

#### We build a handbags-vs-shoes classifier two ways: first training a small CNN entirely from scratch, then freezing a pretrained VGG16 backbone and training only a tiny classification head on top. Live webcam demos before and after show the difference.

---
# 1 · Setup & Data

#### Download the dataset from GitHub and split into train / val / test. Images are resized to 224×224 to match what VGG16 expects.

In [ ]:
import os, pathlib, requests
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

keras.utils.set_random_seed(42)

In [ ]:
API      = 'https://api.github.com/repos/ucla-anderson-SSAI/SSAI/contents/handbags-shoes'
base_dir = pathlib.Path('handbags-shoes')

for category in ('handbags', 'shoes'):
    files = sorted(requests.get(f'{API}/{category}').json(), key=lambda f: f['name'])
    for split, sl in [('train', slice(0,50)), ('validation', slice(50,75)), ('test', slice(75,None))]:
        dst = base_dir / split / category
        os.makedirs(dst, exist_ok=True)
        for f in files[sl]:
            out = dst / f['name']
            if not out.exists():
                out.write_bytes(requests.get(f['download_url']).content)

train_ds = keras.utils.image_dataset_from_directory(base_dir/'train',      image_size=(224,224), batch_size=32, label_mode='binary')
val_ds   = keras.utils.image_dataset_from_directory(base_dir/'validation', image_size=(224,224), batch_size=32, label_mode='binary')
test_ds  = keras.utils.image_dataset_from_directory(base_dir/'test',       image_size=(224,224), batch_size=32, label_mode='binary')

# Quick look at a sample of training images
plt.figure(figsize=(8, 3))
for imgs, labels in train_ds.take(1):
    for i in range(6):
        plt.subplot(1, 6, i+1)
        plt.imshow(imgs[i].numpy().astype('uint8'))
        plt.title('shoe' if labels[i]==1 else 'handbag', fontsize=9)
        plt.axis('off')
plt.tight_layout(); plt.show()

---
# 2 · Webcam Helper

#### Sets up `live_detect()` — a rolling webcam loop that grabs frames, classifies each one, and updates the display in place. We'll reuse this before and after transfer learning.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import time

def live_detect(model_fn, title='Live Detection', n_frames=60, interval=0.1):
    """Stream webcam frames into a model and display rolling predictions.
    model_fn : callable(arr) -> float probability of 'shoe'
    n_frames : how many frames to capture before stopping
    interval : seconds between frames (tune to your GPU/CPU speed)
    """
    # Start the webcam and expose a JS function to grab single frames
    js_setup = Javascript('''
        window._stream = null;
        window._video  = null;

        async function startCam() {
            if (window._stream) return 'already running';
            const div   = document.createElement('div');
            div.id      = 'cam-container';
            div.style.cssText = 'padding:8px;background:#111;display:inline-block;border-radius:8px;';
            const label = document.createElement('div');
            label.id    = 'cam-label';
            label.style.cssText = 'color:#fff;font-family:monospace;font-size:14px;margin-bottom:4px;';
            label.textContent = 'Starting...';
            const video = document.createElement('video');
            video.style.cssText = 'display:block;border-radius:4px;';
            div.appendChild(label);
            div.appendChild(video);
            document.body.appendChild(div);
            window._video = video;
            window._stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = window._stream;
            await video.play();
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            return 'started';
        }

        async function grabFrame() {
            const video  = window._video;
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth; canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            return canvas.toDataURL('image/jpeg', 0.7);
        }

        function updateLabel(text) {
            const el = document.getElementById('cam-label');
            if (el) el.textContent = text;
        }

        function stopCam() {
            if (window._stream) {
                window._stream.getTracks().forEach(t => t.stop());
                window._stream = null;
            }
            const div = document.getElementById('cam-container');
            if (div) div.remove();
        }
    ''')
    display(js_setup)
    eval_js('startCam()')

    print(f'{title} — running for {n_frames} frames. Results appear below.')
    result_display = display('', display_id=True)

    for i in range(n_frames):
        # Grab a frame from the browser
        data   = eval_js('grabFrame()')
        img_bytes = b64decode(data.split(',')[1])

        # Decode and classify
        import io
        from PIL import Image as PILImage
        img    = PILImage.open(io.BytesIO(img_bytes)).resize((224, 224))
        arr    = np.expand_dims(np.array(img), axis=0).astype('float32')
        p_shoe = model_fn(arr)
        probs  = {'handbag': 1 - p_shoe, 'shoe': p_shoe}
        top    = max(probs, key=probs.get)
        conf   = probs[top]

        # Update the label overlay in the browser
        label_text = f'{title}  |  frame {i+1}/{n_frames}  |  → {top.upper()} ({conf:.1%})'
        eval_js(f'updateLabel({json.dumps(label_text)})')

        # Also update a matplotlib frame in the notebook output
        fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(8, 3))
        ax_img.imshow(img); ax_img.axis('off')
        ax_img.set_title(f'→ {top.upper()} ({conf:.1%})', fontsize=11)
        names = sorted(probs, key=probs.get, reverse=True)
        vals  = [probs[n] for n in names]
        ax_bar.barh(names, vals, color=['#3498db','#2980b9'])
        ax_bar.invert_yaxis(); ax_bar.set_xlim(0,1)
        ax_bar.set_xlabel('Confidence')
        ax_bar.set_title(title, fontsize=10)
        for j, v in enumerate(vals):
            ax_bar.text(min(v+0.02, 0.93), j, f'{v:.1%}', va='center', fontsize=9)
        plt.tight_layout()

        # Overwrite the previous frame in-place
        result_display.update(fig)
        plt.close(fig)
        time.sleep(interval)

    eval_js('stopCam()')
    print('Done.')

import json  # needed for eval_js label update

---
# 3 · Baseline: CNN Trained from Scratch

#### A small CNN with randomly initialised weights, trained on just 100 images. With so little data and no prior knowledge, it tends to struggle — this sets the bar we want to beat.

In [ ]:
baseline_cnn = models.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='baseline_cnn')

baseline_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
baseline_cnn.fit(train_ds, epochs=10, validation_data=val_ds, verbose=0)

acc_baseline = baseline_cnn.evaluate(test_ds, verbose=0)[1]
print(f'Baseline CNN test accuracy: {acc_baseline:.4f}')

---
# 4 · Live Detection — Before Transfer Learning

#### Point your webcam at a handbag or shoe and capture. This is the baseline CNN making a cold guess with no pretrained knowledge.

In [ ]:
def baseline_predict(arr):
    return float(baseline_cnn.predict(arr, verbose=0)[0][0])

try:
    live_detect(baseline_predict, title='Before Transfer Learning (Baseline CNN)')
except Exception as err:
    print(err)

---
# 5 · Transfer Learning

#### We freeze VGG16's ImageNet-trained convolutional base and use it as a fixed feature extractor. Every image is passed through once to produce 7×7×512 feature maps, then we train a small dense head on top of those features. The backbone never changes — only the head learns.

In [ ]:
# Load VGG16 without its classifier top; freeze all weights
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

# Extract features once and cache — makes head training very fast
def extract_features(ds):
    feats, labs = [], []
    for imgs, y in ds:
        feats.append(vgg_base.predict(preprocess_input(imgs), verbose=0))
        labs.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labs)

train_feats, train_labels = extract_features(train_ds)
val_feats,   val_labels   = extract_features(val_ds)
test_feats,  test_labels  = extract_features(test_ds)
print(f'Feature shape: {train_feats.shape}  (samples × 7 × 7 × 512)')

In [ ]:
# Small classification head trained on top of the frozen features
tl_head = models.Sequential([
    layers.Flatten(input_shape=(7, 7, 512)),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='transfer_learning_head')

tl_head.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
tl_head.fit(train_feats, train_labels, epochs=10, batch_size=32,
            validation_data=(val_feats, val_labels), verbose=0)

acc_tl = tl_head.evaluate(test_feats, test_labels, verbose=0)[1]
print(f'Transfer learning test accuracy: {acc_tl:.4f}')

# Quick comparison
plt.bar(['Baseline CNN', 'Transfer Learning'], [acc_baseline, acc_tl], color=['#c0392b', '#2980b9'])
plt.ylabel('Test Accuracy'); plt.ylim(0.4, 1.02)
plt.title('Baseline vs Transfer Learning')
plt.show()

---
# 6 · Live Detection — After Transfer Learning

#### Same webcam demo, now using the transfer learning model. Compare its confidence against what you saw in Section 4.

In [ ]:
def tl_predict(arr):
    feats = vgg_base.predict(preprocess_input(arr.copy()), verbose=0)
    return float(tl_head.predict(feats, verbose=0)[0][0])

try:
    live_detect(tl_predict, title='After Transfer Learning (VGG16 backbone)')
except Exception as err:
    print(err)